# STAT163 · Practice 5 — tidy data from two publishers

One notebook across your two practice sessions. Submit the changes in it.

Both tables here are files as their publishers put them on the web: the World Bank and
Eurostat, the statistical office of the European Union. A table is **tidy** when each
variable is a column and each observation is a row. Neither of these two is. You make each
one tidy with `melt`, and with `pivot_table` you build wide tables for reading.

Each part has **Task** cells with a self-check under them. They are the required minimum,
and they carry the grade: a Task cell earns its marks by holding an answer that runs. Each
part ends with an **Explore and discuss** block, which is optional. The note in Part 3 and
the disclosure cell get feedback.

**AI in this practice: recommended mode.**

Try each task yourself first.

If you are stuck in the classroom, ask the teacher or TA. Working remotely: ask your LLM to
explain the concept rather than to solve. Two boundaries are firm:

- say what you used, in the disclosure cell at the end
- do not submit a notebook authored end to end by AI. It is not your work and it is not
  accepted.

## Before you start

Replace the placeholder with your real name and run the cell.

In [1]:
from selfcheck import check, check_col, check_identity

student_name = "artem avramenko"   # ← replace with your name

check_identity(student_name)

✅ student_name = "artem avramenko"


### `melt` and `pivot_table` on a small table

Below are the medals that three countries won at the Paris 2024 Olympic Games: one row per
country and one column per kind of medal. The notebook `01-tidy-data.ipynb` of the `week5-pre-lecture`
repository has more worked examples.

In [2]:
import pandas as pd

medals = pd.DataFrame({
    "country": ["United States", "China", "Japan"],
    "gold": [40, 40, 20],
    "silver": [44, 27, 12],
    "bronze": [42, 24, 13],
})
medals

,country,gold,silver,bronze
0,United States,40,44,42
1,China,40,27,24
2,Japan,20,12,13


With `melt` you make a table longer: the column headers become the values of one column. In
`id_vars` you name the columns that stay as they are, in `var_name` the new column for the
old headers, and in `value_name` the new column for the values.

In [3]:
medals_long = medals.melt(id_vars="country", var_name="medal", value_name="count")
medals_long

,country,medal,count
0,United States,gold,40
1,China,gold,40
2,Japan,gold,20
3,United States,silver,44
4,China,silver,27
5,Japan,silver,12
6,United States,bronze,42
7,China,bronze,24
8,Japan,bronze,13


One row is now one country and one kind of medal.

With `pivot_table` you do the opposite: the values of one column become column headers. In
`index` you name the columns that identify one row of the result, in `columns` the column
whose values become the new headers, and in `values` the column that fills them.

In [5]:
medals_long.pivot_table(index="country", columns="medal", values="count").reset_index()

medal,country,bronze,gold,silver
0,China,24.0,40.0,27.0
1,Japan,13.0,20.0,12.0
2,United States,42.0,40.0,44.0


The new columns and the rows come out sorted by name, and the counts as decimals. Each cell
holds a mean: here the mean of one row. A mean is a decimal number, so 24
becomes 24.0. With `aggfunc="sum"` you get a sum instead.

The values of `country` are now the row labels: with `.loc` you select rows by them, and
with `reset_index()` you turn them back into an ordinary column.

## Part 1 — World Bank population by country

The World Bank publishes the population of each country for every year since 1960. Each
number is an estimate of how many people live in the country in the middle of that year,
whatever their citizenship. The file
in `data/` is the CSV from the indicator's page,
[data.worldbank.org/indicator/SP.POP.TOTL](https://data.worldbank.org/indicator/SP.POP.TOTL),
as it downloads. It starts with four lines about the source before the header row, and with
`skiprows=4` you skip them.

In [6]:
wb = pd.read_csv("data/world_bank_population.csv", skiprows=4)

wb.head()

,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
0,Aruba,ABW,"Population, total",SP.POP.TOTL,54922.0,55578.0,56320.0,57002.0,57619.0,58190.0,...,108735.0,108908.0,109203.0,108587.0,107700.0,107310.0,107359.0,107995.0,108785.0,NaN
1,Africa Eastern and Southern,AFE,"Population, total",SP.POP.TOTL,130075728.0,133534923.0,137171659.0,140945536.0,144904094.0,149033472.0,...,640058741.0,657801085.0,675950189.0,694446100.0,713090928.0,731821393.0,750491370.0,769280888.0,788844284.0,NaN
2,Afghanistan,AFG,"Population, total",SP.POP.TOTL,9035043.0,9214083.0,9404406.0,9604487.0,9814318.0,10036008.0,...,35688935.0,36743039.0,37856121.0,39068979.0,40000412.0,40578842.0,41454761.0,42647492.0,43844111.0,NaN
3,Africa Western and Central,AFW,"Population, total",SP.POP.TOTL,97630925.0,99706674.0,101854756.0,104089175.0,106384410.0,108754449.0,...,440150152.0,451395343.0,462522286.0,473687685.0,484978794.0,496366058.0,508318102.0,520655398.0,532809933.0,NaN
4,Angola,AGO,"Population, total",SP.POP.TOTL,5231654.0,5301583.0,5354310.0,5408320.0,5464187.0,5521981.0,...,30234839.0,31297155.0,32375632.0,33451132.0,34532429.0,35635029.0,36749906.0,37885849.0,39040039.0,NaN


**What is one row, and where are the years?**

**Which columns hold nothing useful?** Count the distinct values in each column that is not
a year.

**Which column says the same thing as another one?**

In [6]:
# Sandbox

**Task 1.1: make `pop_long`, one row per country and year.**

- drop the columns that hold nothing useful, and the one that repeats another
- melt the years into one column, `year`, and their numbers into `population`
- make `year` a number
- drop the rows with no population

*You may need:* `drop(columns=...)`, `melt`, `astype(int)`, `dropna`.

In [8]:
# Sandbox
clean = wb.drop(columns=['Country Code', 'Indicator Name', 'Indicator Code', 'Unnamed: 70'])
clean.melt(id_vars="Country Name", var_name="year", value_name="population")

,Country Name,year,population
0,Aruba,1960,54922.0
1,Africa Eastern and Southern,1960,130075728.0
2,Afghanistan,1960,9035043.0
3,Africa Western and Central,1960,97630925.0
4,Angola,1960,5231654.0
...,...,...,...
17485,Kosovo,2025,1576876.0
17486,"Yemen, Rep.",2025,41773878.0
17487,South Africa,2025,64747319.0
17488,Zambia,2025,21913874.0


In [10]:
# Task 1.1 — pop_long: one row per country and year, with the columns Country Name,
# year (a number) and population. No row without a population.
pop_long = clean.melt(id_vars="Country Name", var_name="year", value_name="population")
pop_long.year = pop_long.year.astype(int)
pop_long.dropna()

check("pop_long", pop_long, "DataFrame")
check_col(pop_long, "year")
check_col(pop_long, "population")
pop_long

✅ pop_long = a DataFrame of 17,490 rows and 3 columns
✅ ["year"] created — 17,490 values, dtype int64, first one is 1960
✅ ["population"] created — 17,490 values, dtype float64, first one is 54922.0


,Country Name,year,population
0,Aruba,1960,54922.0
1,Africa Eastern and Southern,1960,130075728.0
2,Afghanistan,1960,9035043.0
3,Africa Western and Central,1960,97630925.0
4,Angola,1960,5231654.0
...,...,...,...
17485,Kosovo,2025,1576876.0
17486,"Yemen, Rep.",2025,41773878.0
17487,South Africa,2025,64747319.0
17488,Zambia,2025,21913874.0


**Is every row a country?** Sort the 2025 rows by population and read the top twenty
names.

In [12]:
# Sandbox
pop_long[pop_long['Country Name'] == 'Ukraine']

,Country Name,year,population
247,Ukraine,1960,42824093.0
512,Ukraine,1961,43399766.0
777,Ukraine,1962,43943674.0
1042,Ukraine,1963,44462372.0
1307,Ukraine,1964,44951156.0
...,...,...,...
16412,Ukraine,2021,44298640.0
16677,Ukraine,2022,41048766.0
16942,Ukraine,2023,37732836.0
17207,Ukraine,2024,37860221.0


**What would you need to count only the countries?**

A table for people to read is often wide, and you build one from `pop_long` now.

**Task 1.2: make `neighbours`, the population of Ukraine and six of its neighbours, with one row
per year and one column per country.**

*You may need:* `pivot_table`, `.loc`.

In [14]:
# Sandbox
countries = ["Ukraine", "Poland", "Slovak Republic", "Hungary", "Romania", "Moldova", "Belarus"]
years = [1990, 2000, 2010, 2020, 2025]
pop_long.pivot_table(values='population', index="Country Name", columns="year").loc[countries, years]


year,1990,2000,2010,2020,2025
Country Name,,,,,
Ukraine,52054092.0,49556660.0,46456003.0,44680014.0,38980376.0
Poland,38110782.0,38258629.0,38042794.0,37515748.0,36435861.0
Slovak Republic,5299187.0,5388720.0,5391428.0,5458827.0,5413813.0
Hungary,10373988.0,10210971.0,10000023.0,9670419.0,9514251.0
Romania,23201835.0,22442971.0,20246871.0,19265250.0,19020271.0
Moldova,2977912.0,2924668.0,2862354.0,2635130.0,2360527.0
Belarus,10189348.0,9979610.0,9483836.0,9379952.0,9085991.0


In [15]:
# Task 1.2 — neighbours: from pop_long, the rows of these countries and years,
# with one row per year and one column per country.
countries = ["Ukraine", "Poland", "Slovak Republic", "Hungary", "Romania", "Moldova", "Belarus"]
years = [1990, 2000, 2010, 2020, 2025]
neighbours = pop_long.pivot_table(values='population', index="Country Name", columns="year").loc[countries, years]

check("neighbours", neighbours, "DataFrame")

✅ neighbours = a DataFrame of 7 rows and 5 columns


### Explore and discuss

**A chart, optional.** If you worked through the charts notebook this week, draw the
population of the seven countries for every year from 1990, one line per country. Which
table do you draw it from, `pop_long` or `neighbours`, and why?

**Discuss:** Ukraine's number for 2025 is lower than for 2020 by more than five million.
Before you report that fall, what would you check about how the number was estimated?
The indicator's page names its sources.

In [ ]:
# Explore

## Part 2 — Eurostat unemployment by age and sex

Eurostat publishes the unemployment rate of each European country per year, by age and sex.
The file in `data/` is the dataset `une_rt_a` as the Eurostat database sends it in TSV
format: values separated by tabs rather than commas. With `sep="\t"` you tell `read_csv` so.

In [16]:
une = pd.read_csv("data/une_rt_a.tsv", sep="\t")

une.head()

,"freq,age,unit,sex,geo\TIME_PERIOD",2003,2004,2005,2006,2007,2008,2009,2010,2011,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,"A,Y15-24,PC_ACT,F,AT",:,:,:,:,:,:,10.5,9.8,9.5,...,10.7,9.1,9.8,8.1,10.6,11.3,9.5,10.7,10.1,10.8
1,"A,Y15-24,PC_ACT,F,BA",:,:,:,:,:,:,:,:,:,...,:,:,:,:,:,44.5,42.1,33.2,35.6,37.4
2,"A,Y15-24,PC_ACT,F,BE",:,:,:,:,:,:,22.6,22.4,18.9,...,18.3,18.2 b,15.3,12.3,15.6,16.2,14.4,14.4,14.8,15.7
3,"A,Y15-24,PC_ACT,F,BG",:,:,:,:,:,:,17.8,24.4 b,28.1 b,...,21.1,16.3,15.7,11.8 bu,17.7 u,15.1 u,10.4 u,11.1 u,11.7 bu,11.8
4,"A,Y15-24,PC_ACT,F,CH",:,:,:,:,:,:,:,8.7 b,7.9,...,8.4,8.0,7.5,7.2,8.0,8.3,6.8,7.6,8.5,9.2


**Read the name of the first column and the first values under it. How many variables does
this one column hold?**

**What do the values in the year columns look like?** Read a few rows and list what you find
besides numbers.

In [ ]:
# Sandbox

The name of the first column holds a backslash, which is awkward to type in Python code, so
we rename it to `codes`.

In [18]:
une = une.rename(columns={une.columns[0]: "codes"})
une

,codes,2003,2004,2005,2006,2007,2008,2009,2010,2011,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,"A,Y15-24,PC_ACT,F,AT",:,:,:,:,:,:,10.5,9.8,9.5,...,10.7,9.1,9.8,8.1,10.6,11.3,9.5,10.7,10.1,10.8
1,"A,Y15-24,PC_ACT,F,BA",:,:,:,:,:,:,:,:,:,...,:,:,:,:,:,44.5,42.1,33.2,35.6,37.4
2,"A,Y15-24,PC_ACT,F,BE",:,:,:,:,:,:,22.6,22.4,18.9,...,18.3,18.2 b,15.3,12.3,15.6,16.2,14.4,14.4,14.8,15.7
3,"A,Y15-24,PC_ACT,F,BG",:,:,:,:,:,:,17.8,24.4 b,28.1 b,...,21.1,16.3,15.7,11.8 bu,17.7 u,15.1 u,10.4 u,11.1 u,11.7 bu,11.8
4,"A,Y15-24,PC_ACT,F,CH",:,:,:,:,:,:,:,8.7 b,7.9,...,8.4,8.0,7.5,7.2,8.0,8.3,6.8,7.6,8.5,9.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2389,"A,Y55-74,THS_PER,T,RS",:,:,:,:,:,:,:,54,69,...,46,45,47,38,29,39,41,42,42,37
2390,"A,Y55-74,THS_PER,T,SE",:,:,40,41,37,36,50,54,50,...,52,52,47,50,63,77,64,65,73,81
2391,"A,Y55-74,THS_PER,T,SI",:,:,:,:,:,:,3 u,4 u,6,...,8,9,7,7,6,7,7,7,7,7
2392,"A,Y55-74,THS_PER,T,SK",:,:,:,:,:,:,21,31,32 b,...,37,26,23,21,22,25,22,19,16,21


Each value of `codes` has five parts separated by commas, in this order: `freq`, `age` (the age group, `Y15-74` is 15 to 74), `unit`, `sex`
(`F`, `M`, and `T` for total) and `geo` (Eurostat's country code, for example `EL` for
Greece).

**Task 2.1: make `une_long`, one row per code combination and year.**

- melt the years into one column, `year`, and the cells into `value`
- split `codes` into five columns: with `une_long[["freq", "age", "unit", "sex", "geo"]] = ...`
  you add all five in one step
- make `year` a number

*You may need:* `melt`, `.str.split(",", expand=True)`, `astype(int)`.

In [19]:
# Sandbox
une.melt(id_vars='codes', var_name='year', value_name='value')

,codes,year,value
0,"A,Y15-24,PC_ACT,F,AT",2003,:
1,"A,Y15-24,PC_ACT,F,BA",2003,:
2,"A,Y15-24,PC_ACT,F,BE",2003,:
3,"A,Y15-24,PC_ACT,F,BG",2003,:
4,"A,Y15-24,PC_ACT,F,CH",2003,:
...,...,...,...
55057,"A,Y55-74,THS_PER,T,RS",2025,37
55058,"A,Y55-74,THS_PER,T,SE",2025,81
55059,"A,Y55-74,THS_PER,T,SI",2025,7
55060,"A,Y55-74,THS_PER,T,SK",2025,21


In [27]:
# Task 2.1 — une_long: one row per code combination and year, with the columns freq,
# age, unit, sex, geo, year (a number) and value (the text of the cell, as in the file).
une_long = une.melt(id_vars='codes', var_name='year', value_name='value')
une_long[["freq", "age", "unit", "sex", "geo"]] = une_long.codes.str.split(",", expand=True)
une_long.year = une_long.year.astype(int)

check("une_long", une_long, "DataFrame")
check_col(une_long, "geo")
check_col(une_long, "year")
une_long.head()


✅ une_long = a DataFrame of 55,062 rows and 8 columns
✅ ["geo"] created — 55,062 values, dtype str, first one is AT
✅ ["year"] created — 55,062 values, dtype int64, first one is 2003


,codes,year,value,freq,age,unit,sex,geo
0,"A,Y15-24,PC_ACT,F,AT",2003,:,A,Y15-24,PC_ACT,F,AT
1,"A,Y15-24,PC_ACT,F,BA",2003,:,A,Y15-24,PC_ACT,F,BA
2,"A,Y15-24,PC_ACT,F,BE",2003,:,A,Y15-24,PC_ACT,F,BE
3,"A,Y15-24,PC_ACT,F,BG",2003,:,A,Y15-24,PC_ACT,F,BG
4,"A,Y15-24,PC_ACT,F,CH",2003,:,A,Y15-24,PC_ACT,F,CH


**Which of the five new columns holds only one value?**

**List the values of `age`. Which groups overlap, and what happens to a sum over all ages?**

In [26]:
# Sandbox
une_long.age.unique()

<StringArray>
['Y15-24', 'Y15-29', 'Y15-74', 'Y20-64', 'Y25-54', 'Y25-74', 'Y55-74']
Length: 7, dtype: str

The `value` column still holds text.

**Turn `value` into numbers with `pd.to_numeric`.** If it stops with an error, read the
error. Then run it again with `errors="coerce"`: with it you get a missing value in place of
each text that is not a number, instead of an error.

**How many values are missing now, and which texts became missing?** Compare it with how
many cells hold `:`. Look at exactly what those cells hold before you count.

In [ ]:
# Sandbox

The letters after some numbers are **flags**, notes from Eurostat about the value: `b` a
break in the series (the way of measuring changed that year), `d` a definition that differs,
`u` low reliability. A value can carry two flags at once: `bu` is `b` and `u`.

**Task 2.2: split `value` into two columns, `number` for the number and `flag` for the
letters.**

*You may need:* `.str.split(expand=True)`, `pd.to_numeric`.

In [ ]:
# Sandbox

In [34]:
# Task 2.2 — two columns on une_long: number, the number in value as a float (missing
# where there is none), and flag, the letters after it (missing where there are none).
une_long["number"] = une_long.value.str.split(expand=True)[0]
une_long["flag"] = une_long.value.str.split(expand=True)[1]
une_long.number = pd.to_numeric(une_long.number, errors='coerce')
check_col(une_long, "number")
check_col(une_long, "flag")
une_long

✅ ["number"] created — 55,062 values, dtype float64, first one is nan
✅ ["flag"] created — 55,062 values, dtype str, first one is nan


,codes,year,value,freq,age,unit,sex,geo,number,flag
0,"A,Y15-24,PC_ACT,F,AT",2003,:,A,Y15-24,PC_ACT,F,AT,NaN,NaN
1,"A,Y15-24,PC_ACT,F,BA",2003,:,A,Y15-24,PC_ACT,F,BA,NaN,NaN
2,"A,Y15-24,PC_ACT,F,BE",2003,:,A,Y15-24,PC_ACT,F,BE,NaN,NaN
3,"A,Y15-24,PC_ACT,F,BG",2003,:,A,Y15-24,PC_ACT,F,BG,NaN,NaN
4,"A,Y15-24,PC_ACT,F,CH",2003,:,A,Y15-24,PC_ACT,F,CH,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
55057,"A,Y55-74,THS_PER,T,RS",2025,37,A,Y55-74,THS_PER,T,RS,37.0,NaN
55058,"A,Y55-74,THS_PER,T,SE",2025,81,A,Y55-74,THS_PER,T,SE,81.0,NaN
55059,"A,Y55-74,THS_PER,T,SI",2025,7,A,Y55-74,THS_PER,T,SI,7.0,NaN
55060,"A,Y55-74,THS_PER,T,SK",2025,21,A,Y55-74,THS_PER,T,SK,21.0,NaN


**Can you compare any two values in `number`?** Count the rows per `unit`, and read the
[dataset's page](https://ec.europa.eu/eurostat/databrowser/view/une_rt_a/default/table)
for what the three units are.

In [ ]:
# Sandbox

**Task 2.3: make `by_sex`, the rates of women and men side by side, and
`widest_gap_2025`, the country with the widest gap in 2025.**

- keep one unit, **`PC_ACT`**: the unemployed as a percentage of the labour force, which is
  everyone who has a job or is looking for one
- keep one age group, **`Y15-74`**
- put the three values of `sex` side by side, and add `gap`, women minus men

*You may need:* `pivot_table` with two columns in `index`, `.loc`, `idxmax`.

In [38]:
# Sandbox
une_long_new = une_long[(une_long.unit == 'PC_ACT') & (une_long.age == 'Y15-74')]
une_long_new.pivot_table(index=['year', 'geo'], columns='sex', values='number')

sex          F    M    T
year geo                
2003 FR    9.3  7.8  8.5
2004 FR    9.7  8.2  8.9
2005 FR    9.6  8.2  8.9
     SE    7.9  7.9  7.9
2006 FR    9.5  8.3  8.9
...        ...  ...  ...
2025 RS    9.1  8.4  8.7
     SE    8.8  8.9  8.8
     SI    4.3  3.6  3.9
     SK    5.8  5.1  5.4
     TR   11.4  6.9  8.4

[635 rows x 3 columns]

In [50]:
# Task 2.3 — by_sex: the rows of unit PC_ACT and age Y15-74, one row per year and geo,
# with the columns F, M and T, and a column gap: F minus M.
# Then widest_gap_2025: the geo code with the largest gap in 2025.
by_sex = une_long_new.pivot_table(index=['year', 'geo'], columns='sex', values='number')
by_sex["gap"] = by_sex.F - by_sex.M
widest_gap_2025 = by_sex.loc[2025, 'gap'].idxmax()

check("by_sex", by_sex, "DataFrame")
check_col(by_sex, "gap")
check("widest_gap_2025", widest_gap_2025, "text")

✅ by_sex = a DataFrame of 635 rows and 4 columns
✅ ["gap"] created — 635 values, dtype float64, first one is 1.5000000000000009
✅ widest_gap_2025 = "BA"


### Explore and discuss

**Countries and groups of countries.** List the values of `geo`. Which of them are not a
country?

**Is `T` the mean of `F` and `M`?** Compare `T` with the mean of the two, row by row. When
is it closer to `F`, and when to `M`? What would decide it?

**Flags.** Count the flags per country. Would you report a rate that carries `u`?

**Discuss:** in which countries is unemployment higher for women, and in which for men?
Before you compare two countries, which flags would you look at?

In [ ]:
# Explore

## Part 3 — The note

**A colleague asks: how many young people, aged 15 to 24, were unemployed in the EU in
2025?**

Find the answer in `une_long`. Then write it in the cell below, with one line for each choice
you made on the way to it, and run the cell.

In [ ]:
# Sandbox

> **The answer:** *(one number, and what it counts)*
>
> **The choices:** *(one line for each column you filtered on: the value you kept, and why)*

## Before you submit

**Say what you used.** One line naming the tool and the step it helped with, or
"No AI used".

> *Example: "Used Claude to see why astype(float) stopped on one value."*

In [ ]:
ai_disclosure = "..."

check("ai_disclosure", ai_disclosure, "text")

**Submit:** restart the kernel and run every cell from the top, check that every cell runs,
save, commit and push. Then paste your repository URL into the Week 5 practice assignment
on Moodle.

The deadline is shown on the Moodle assignment.